# Sharing one GPU: inference and LoRA training, queued by KAI

One 12 GiB GPU (an RTX A2000, say), three workloads, each with a fixed share of GPU memory:

```text
GPU, 12 GiB
├── Qwen2.5-1.5B inference endpoint   5 GiB   serving the whole time
├── LoRA fine-tune A                  6 GiB   trains beside it
└── LoRA fine-tune B                  6 GiB   does not fit: waits in the queue, starts when A finishes
```

KAI places each workload by its GPU-memory share and holds what does not fit in the project's queue;
HAMi-core caps each container at its share. Each training run downloads the model to its scratch
volume and saves its LoRA adapter to its own checkpoint volume (dynamically provisioned from
Longhorn and kept after the run).

## 1. Connect

In [ ]:
from rancher_ai import Client

ai = Client(project="team-a")
ai.whoami()["user"]

## 2. The inference endpoint already on the GPU

In [ ]:
ep = ai.endpoints.get("qwen-shared")
ep

In [ ]:
import time

def ask(q, n=60):
    t = time.time()
    answer = ep.chat(q, max_tokens=n, print_=False)
    return f"{time.time() - t:.2f}s  {answer}"

ask("In one sentence, what does LoRA fine-tuning change in a model?")

In [ ]:
def gpu_shares():
    """Who holds what on the GPU: every running pod with a KAI GPU-memory share."""
    rows = []
    for p in ai.conn.core.list_namespaced_pod(ai.project).items:
        mib = (p.metadata.annotations or {}).get("gpu-memory")
        if mib and p.status.phase in ("Running", "Pending"):
            rows.append({"pod": p.metadata.name, "share": f"{int(mib) / 1024:.1f} GiB",
                         "state": "on the GPU" if p.spec.node_name else "waiting in the queue"})
    from rancher_ai.display import Table
    return Table(rows, ["pod", "share", "state"])

gpu_shares()

## 3. Submit two LoRA fine-tunes, 6 GiB each

The first fits beside the endpoint (0.42 + 0.51 of the GPU); the second would take the GPU past 1.0, so KAI holds it in the queue.

In [ ]:
TRAIN = dict(
    profile="shared-gpu-dev",
    image="pytorch/pytorch:2.5.1-cuda12.4-cudnn9-runtime",   # has pip, for transformers + peft
    gpu_memory=6,                                             # GiB of the GPU per run
    script=open("lora_train.py").read(),
    runtime_hours=1,
)
env = {"HF_HOME": "/scratch/hf", "STEPS": "300", "SEQ_LEN": "512", "BATCH": "4"}

run_a = ai.runs.create(name="dev-lora-a", env={**env, "RUN_NAME": "dev-lora-a"}, **TRAIN)
run_b = ai.runs.create(name="dev-lora-b", env={**env, "RUN_NAME": "dev-lora-b"}, **TRAIN)
run_a, run_b

In [ ]:
run_a.wait({"Running"}, timeout=900)
time.sleep(5)
run_b.refresh()
print("A:", run_a.state, "   B:", run_b.state)
gpu_shares()

## 4. Training and serving at the same time

While A trains, the endpoint keeps answering.

In [ ]:
for q in ["Name three uses of a GPU in one line.", "What is a learning rate?", "Say hello in French."]:
    print(ask(q, 40))

In [ ]:
run_a.logs(tail=6)

## 5. A finishes, B starts

No one resubmits B: KAI admits it as soon as A's share is released.

In [ ]:
run_a.wait(timeout=1800)
print("A:", run_a.state)
run_b.wait({"Running", "Completed"}, timeout=900)
print("B:", run_b.refresh().state)
gpu_shares()

In [ ]:
run_b.wait(timeout=1800)
print("B:", run_b.state)
run_b.logs(tail=4)

## 6. The adapters, kept on their own volumes

In [ ]:
ai.checkpoints.table()

In [ ]:
print(ai.checkpoints.get("dev-lora-a-checkpoints").ls("dev-lora-a"))

In [ ]:
ai.runs.table()

## 7. Clean up (optional)

Deleting a run keeps its checkpoint volume; delete the volume separately when the adapter is no longer needed.

In [ ]:
# run_a.delete(); run_b.delete()
# ai.checkpoints.delete("dev-lora-a-checkpoints", confirm=True)
# ai.checkpoints.delete("dev-lora-b-checkpoints", confirm=True)